
# Dust attenuation laws across the galaxy zoo

Four named attenuation laws applied to the *same* intrinsic SED at the
*same* V-band optical depth (τ_V = 1.0), illustrating how dust geometry
and grain-size composition vary across the local universe.

Shows:
- **Cardelli MW (R_V = 3.1)**: Milky Way extinction curve with prominent
  2175 Å bump (graphite/silicate composite).
- **Calzetti (starburst)**: Modified power-law characteristic of dusty
  starburst galaxies; flattened UV slope, no bump.
- **SMC (Gordon+2003)**: Small Magellanic Cloud extinction curve; steepest
  UV attenuation due to smaller grain size.
- **Kriek & Conroy (2013)**: Empirical composite of Calzetti base and
  Lorentzian UV bump; bridges starburst and extinction-curve regimes.

Key observations: SMC attenuation is steepest at 1000–2000 Å, while
Cardelli shows the characteristic 2175 Å absorption feature. Calzetti
flattens the UV relative to optical, typical of energy-driven star
formation feedback. Kriek & Conroy provides intermediate behavior
combining both regimes.

References: Cardelli et al. (1989), Calzetti et al. (2000),
Gordon et al. (2003), Kriek & Conroy (2013).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Four canonical dust laws spanning the UV-optical attenuation landscape:
# - Cardelli MW: MW extinction with 2175 Å graphite bump (R_V = 3.1)
# - Calzetti: starburst attenuation law, flattened UV slope
# - SMC: Small Magellanic Cloud, steep UV due to small grains
# - Kriek & Conroy: modified Calzetti with bump, empirical composite
LAWS = [
    ("cardelli", "Cardelli+1989 (MW, R_V=3.1)"),
    ("calzetti", "Calzetti+2000 (starburst)"),
    ("smc", "SMC (Gordon+2003)"),
    ("kriek_conroy", "Kriek & Conroy 2013"),
]
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]  # matplotlib tab10 subset

# Fixed SFH and stellar population
SFH = {
    "type": "tsnorm",
    "*": tengri.FIXED,
    "peak_lbt_gyr": 2.0,
    "width_gyr": 1.0,
    "log_total_mass": 10.0,
    "skew": 0.0,
    "trunc": 13.0,
}

# Load default SSP (bare stellar, compatible with all nebular backends)
ssp = tengri.load_ssp()

fig, ax = plt.subplots(figsize=(7.2, 4.6))

# Intrinsic (unreddened) SED as reference
ref_model = tengri.SEDModel.build(
    ssp,
    sfh=SFH,
    dust={
        "type": "two_component",
        "law_diff": "calzetti",
        "*": tengri.FIXED,
        "tau_diff": 0.0,
        "tau_bc": 0.0,
    },
    redshift=tengri.Fixed(0.05),
)
p_ref = dict(ref_model.spec.sample(jax.random.PRNGKey(0)))
sed_ref = np.asarray(ref_model.predict_rest_sed(p_ref).sed)
wave = np.asarray(ref_model.predict_rest_sed(p_ref).wavelength)
C_AA_PER_S = 2.998e18
nu = C_AA_PER_S / wave
ax.loglog(
    wave, nu * sed_ref, color="0.05", lw=2.0, label="intrinsic", zorder=10, ls="--"
)

# Plot reddened SED for each law at fixed tau_V = 1.0
for (law, label), color in zip(LAWS, COLORS):
    model = tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust={
            "type": "two_component",
            "law_diff": law,
            "*": tengri.FIXED,
            "tau_diff": 1.0,  # V-band optical depth = 1.0 (representative starburst)
            "tau_bc": 0.0,  # Only diffuse ISM component
        },
        redshift=tengri.Fixed(0.05),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    sed = np.asarray(model.predict_rest_sed(p).sed)
    ax.loglog(wave, nu * sed, color=color, lw=1.8, label=label)

# Annotate key wavelength features
ax.axvline(2175, color="0.55", lw=0.5, ls=":", alpha=0.7)
ax.text(2175, 1.3e40, "2175 Å bump", fontsize=8, color="0.4", rotation=90, va="bottom", ha="right")

ax.axvline(5500, color="0.7", lw=0.4, ls=":", alpha=0.5)
ax.text(5500, 2.8e43, "V-band", fontsize=7, color="0.5", rotation=90, va="top", ha="left")

ax.set(
    xlim=(900, 3e4),
    ylim=(1e40, 8e43),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
)
ax.legend(frameon=False, fontsize=9, loc="lower right", ncol=1)

fig.tight_layout()
plt.savefig("plot_galactic_zoo_dust_laws.png", dpi=150, bbox_inches="tight")
plt.show()